In [6]:
import pandas as pd

# Load the Excel file
file_path = r"C:\Users\Neg\TCGA_GC_project\All_DEGs_GeneSymbols_for_STRING.xlsx"
df = pd.read_excel(file_path)

# Drop rows with missing symbol or log2FC
df = df.dropna(subset=["symbol", "log2FC_x"])

# Select top 100 upregulated and 100 downregulated genes
upregulated = df[df["log2FC_x"] > 0].sort_values(by="log2FC_x", ascending=False).head(100)
downregulated = df[df["log2FC_x"] < 0].sort_values(by="log2FC_x").head(100)

# Combine and save
top200 = pd.concat([upregulated, downregulated])
top200.to_excel(r"C:\Users\Neg\TCGA_GC_project\Top200_New_DEGs.xlsx", index=False)

print("Top 200 DEGs (100 up + 100 down) saved.")


Top 200 DEGs (100 up + 100 down) saved.


In [10]:
import pandas as pd

# File paths
file1 = r"C:\Users\Neg\TCGA_GC_project\Top200_New_DEGs_from_6500.xlsx"
file2 = r"C:\Users\Neg\TCGA_GC_project\Top200_HighConfidence_DEGs.csv"

# Load the files
df1 = pd.read_excel(file1)
df2 = pd.read_csv(file2)

# Show column names to identify correct gene symbol column
print("File 1 columns:", df1.columns.tolist())
print("File 2 columns:", df2.columns.tolist())


File 1 columns: ['symbol', 'Gene', 'log2FC_x', 'p_value', 'FDR', 'abs_log2FC']
File 2 columns: ['Gene', 'log2FC', 'p_value', 'FDR', 'Avg_Tumor_Expression', 'Avg_Normal_Expression']


In [12]:
import pandas as pd

# File paths
file1 = r"C:\Users\Neg\TCGA_GC_project\Top200_New_DEGs_from_6500.xlsx"
file2 = r"C:\Users\Neg\TCGA_GC_project\Top200_HighConfidence_DEGs.csv"

# Load files
df1 = pd.read_excel(file1)
df2 = pd.read_csv(file2)

# Extract and standardize gene names
genes1 = set(df1["symbol"].astype(str).str.upper())
genes2 = set(df2["Gene"].astype(str).str.upper())

# Compare
common_genes = genes1 & genes2
unique_to_file1 = genes1 - genes2
unique_to_file2 = genes2 - genes1

# Report results
print(f"Total in File 1: {len(genes1)}")
print(f"Total in File 2: {len(genes2)}")
print(f"Common genes: {len(common_genes)}")
print(f"Unique to File 1: {len(unique_to_file1)}")
print(f"Unique to File 2: {len(unique_to_file2)}")

# Optionally save for inspection
pd.DataFrame(sorted(unique_to_file1), columns=["Unique_to_File1"]).to_csv("unique_to_file1.csv", index=False)
pd.DataFrame(sorted(unique_to_file2), columns=["Unique_to_File2"]).to_csv("unique_to_file2.csv", index=False)
pd.DataFrame(sorted(common_genes), columns=["Common_Genes"]).to_csv("common_genes.csv", index=False)


Total in File 1: 200
Total in File 2: 200
Common genes: 0
Unique to File 1: 200
Unique to File 2: 200


In [14]:
genes1 = set(df1["symbol"].astype(str).str.strip().str.upper().str.split(".").str[0])
genes2 = set(df2["Gene"].astype(str).str.strip().str.upper().str.split(".").str[0])


In [16]:
print("File 1 examples:", list(genes1)[:5])
print("File 2 examples:", list(genes2)[:5])


File 1 examples: ['OR52M1', 'MIR205', 'IGKV2-36', 'CYP2C59P', 'LOC122394732']
File 2 examples: ['ENSG00000286447', 'ENSG00000253460', 'ENSG00000198889', 'ENSG00000275216', 'ENSG00000204776']


In [18]:
from mygene import MyGeneInfo
mg = MyGeneInfo()

# Convert File 2's Ensembl IDs to gene symbols
ensembl_ids = list(df2["Gene"].astype(str).str.strip().str.split(".").str[0])
res = mg.querymany(ensembl_ids, scopes='ensembl.gene', fields='symbol', species='human')

# Extract mapped symbols
mapped_df = pd.DataFrame(res)
mapped_df = mapped_df[['query', 'symbol']].dropna()

# Merge to add symbols to df2
df2_annotated = df2.merge(mapped_df, left_on="Gene", right_on="query")


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
5 input query terms found no hit:	['ENSG00000227925', 'ENSG00000275216', 'ENSG00000243944', 'ENSG00000224287', 'ENSG00000237838']


In [20]:
# Extract gene symbols
genes1 = set(df1["symbol"].astype(str).str.strip().str.upper())
genes2 = set(df2_annotated["symbol"].astype(str).str.strip().str.upper())

# Compare
common_genes = genes1 & genes2
only_in_1 = genes1 - genes2
only_in_2 = genes2 - genes1

# Print results
print(f"Total in File 1: {len(genes1)}")
print(f"Total in File 2: {len(genes2)}")
print(f"Common genes: {len(common_genes)}")
print(f"Unique to File 1: {len(only_in_1)}")
print(f"Unique to File 2: {len(only_in_2)}")


Total in File 1: 200
Total in File 2: 0
Common genes: 0
Unique to File 1: 200
Unique to File 2: 0


In [1]:
import pandas as pd
import os

# --- CONFIGURATION ---
input_file = r"C:\Users\Neg\TCGA_GC_project\All_DEGs_GeneSymbols_for_STRING.xlsx"
output_folder = r"C:\Users\Neg\TCGA_GC_project\DEG_SPLITS"
genes_column = "Gene"  # Update if the column name is different
chunk_size = 500

# --- Ensure output folder exists ---
os.makedirs(output_folder, exist_ok=True)

# --- Load gene list ---
df = pd.read_excel(input_file)
genes = df[genes_column].dropna().unique()

# --- Split and save chunks ---
for i in range(0, len(genes), chunk_size):
    chunk = genes[i:i+chunk_size]
    part_num = i // chunk_size + 1
    chunk_df = pd.DataFrame(chunk, columns=[genes_column])
    chunk_df.to_excel(os.path.join(output_folder, f"DEGs_part{part_num}.xlsx"), index=False)

print(f"✅ Done! Split into {len(genes) // chunk_size + 1} files in: {output_folder}")


✅ Done! Split into 20 files in: C:\Users\Neg\TCGA_GC_project\DEG_SPLITS


In [ ]:
h